<a href="https://colab.research.google.com/github/AlanChi0720/bio_ai/blob/main/B2_finetune_thermostability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track B — Notebook 2: Fine-tuning ESM-2 to Predict Thermostability

**Big idea:** In B1 we used ESM-2 as a *frozen* feature extractor. Now we'll **fine-tune** the model — actually update some of its weights with gradient descent on our task. This is a step closer to how state-of-the-art protein ML papers train their models.

**Task (regression):** Predict a protein's melting temperature (Tm) from its amino acid sequence.

**Data:** FLIP Meltome Atlas (Jarzab et al. 2020) — thousands of proteins with experimentally measured Tm.

**Two experiments to compare:**
1. **Frozen ESM-2 + regression head** — just train the head
2. **Fine-tuned ESM-2 (last 2 layers) + regression head** — let the model adapt to thermostability

**Estimated time:** ~3-4 hours, GPU **required**.

In [ ]:
!pip install -q transformers torch

In [ ]:
import io, urllib.request, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import EsmTokenizer, EsmModel
from scipy.stats import spearmanr, pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'GPU required for this notebook. Enable GPU in Colab: Runtime → Change runtime type.'
sns.set_theme(style='whitegrid')

## 1. Download FLIP Meltome Data

The FLIP benchmark (Dallago et al. 2021) packages the Meltome Atlas as a CSV with sequences, melting temperatures, and a recommended train/test split. We'll subsample for tractability.

In [ ]:
import pandas as pd
import urllib.request
import zipfile
import io

# FLIP meltome — 正確路徑是 splits.zip，不是直接 CSV
FLIP_ZIP_URL = 'https://github.com/J-SNACKKB/FLIP/raw/main/splits/meltome/splits.zip'

try:
    with urllib.request.urlopen(FLIP_ZIP_URL) as response:
        zip_bytes = response.read()

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        print("zip 內的檔案：", z.namelist())
        # mixed split 對應的檔名
        with z.open('splits/mixed_split.csv') as f:
            df = pd.read_csv(f)

    print(f'Downloaded {len(df)} entries from FLIP meltome mixed split.')

except Exception as e:
    print('Failed:', e)

In [ ]:
df.head()

In [ ]:
# Inspect the data
print('Columns:', df.columns.tolist())
print('Set values:', df['set'].value_counts().to_dict() if 'set' in df.columns else 'no split column')
print(f'Tm range: {df["target"].min():.1f} to {df["target"].max():.1f} C')

fig, ax = plt.subplots(figsize=(6, 3))
sns.histplot(df['target'], bins=40, ax=ax)
ax.set_xlabel('Melting temperature (C)')
ax.set_title('Tm distribution')
plt.tight_layout(); plt.show()

In [ ]:
# Subsample for Colab tractability
MAX_LEN = 400        # truncate long sequences (most important step for memory)
N_TRAIN = 2000
N_TEST  = 500

# Filter by length
df_short = df[df['sequence'].str.len() <= MAX_LEN].reset_index(drop=True)
print(f'After length filter: {len(df_short)}')

# Use FLIP's pre-defined train/test split if available
if 'set' in df.columns:
    train_df = df_short[df_short['set'] == 'train'].sample(min(N_TRAIN, (df_short['set']=='train').sum()), random_state=42)
    test_df  = df_short[df_short['set'] == 'test'].sample(min(N_TEST,  (df_short['set']=='test').sum()),  random_state=42)
else:
    train_df, test_df = train_test_split(df_short, test_size=0.2, random_state=42)
    train_df = train_df.iloc[:N_TRAIN]
    test_df  = test_df.iloc[:N_TEST]

print(f'Train: {len(train_df)},  Test: {len(test_df)}')

## 2. Build PyTorch Dataset and DataLoader

For real fine-tuning we batch sequences and shuffle each epoch. PyTorch's `Dataset` + `DataLoader` is the standard pattern.

In [ ]:
MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)

class ProteinTmDataset(Dataset):
    def __init__(self, df):
        self.seqs = df['sequence'].tolist() # 蛋白質序列
        self.tms = df['target'].astype(np.float32).values # 熔點 Tm 值（目標）
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i], self.tms[i]

def collate(batch):
    seqs = [b[0] for b in batch]
    tms  = torch.tensor([b[1] for b in batch], dtype=torch.float32)
    enc = tokenizer(seqs, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)
    return enc, tms

train_ds = ProteinTmDataset(train_df)
test_ds  = ProteinTmDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, collate_fn=collate)
print('Dataset ready.')

## 3. Define the Model

Architecture: ESM-2 base + a small regression head on top of the mean-pooled embedding.

In [ ]:
class ESM2Regressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, hidden=128, n_unfrozen_layers=0):
        super().__init__()
        self.esm = EsmModel.from_pretrained(model_name)
        d = self.esm.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 1),
        )
        # Freeze all ESM params first, then unfreeze the last N transformer layers
        for p in self.esm.parameters():
            p.requires_grad = False
        if n_unfrozen_layers > 0:
            n_layers = len(self.esm.encoder.layer)
            for layer in self.esm.encoder.layer[n_layers - n_unfrozen_layers:]:
                for p in layer.parameters():
                    p.requires_grad = True

    def forward(self, enc):
        h = self.esm(**enc).last_hidden_state                 # (B, L, D)
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return self.head(pooled).squeeze(-1)

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4. Generic Training Loop

We'll reuse this for both the frozen and fine-tuned experiments.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for enc, tms in loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        p = model(enc).cpu().numpy()
        preds.append(p); trues.append(tms.numpy())
    preds = np.concatenate(preds); trues = np.concatenate(trues)
    rmse = np.sqrt(mean_squared_error(trues, preds))
    rho = spearmanr(trues, preds).correlation
    r = pearsonr(trues, preds)[0]
    return preds, trues, rmse, rho, r

def train(model, n_epochs=5, lr=1e-3):
    print(f'Trainable params: {count_trainable(model):,}')
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(n_epochs):
        model.train(); t0 = time.time(); losses = []
        for enc, tms in train_loader:
            enc = {k: v.to(device) for k, v in enc.items()}
            tms = tms.to(device)
            opt.zero_grad()
            loss = loss_fn(model(enc), tms)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        _, _, rmse, rho, r = evaluate(model, test_loader)
        history.append({'epoch': epoch, 'train_mse': np.mean(losses),
                        'test_rmse': rmse, 'test_spearman': rho, 'test_pearson': r})
        print(f'epoch {epoch}: train_mse={np.mean(losses):.2f}  test_rmse={rmse:.2f}  '
              f'spearman={rho:.3f}  ({time.time()-t0:.0f}s)')
    return pd.DataFrame(history)

## 5. Experiment 1 — Frozen ESM-2 + Trainable Head

Same idea as B1, but a regression task. Only the head's parameters are updated.

In [ ]:
torch.manual_seed(42)
model_frozen = ESM2Regressor(n_unfrozen_layers=0).to(device)
history_frozen = train(model_frozen, n_epochs=5, lr=1e-3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(history_frozen['epoch'], history_frozen['train_mse'], marker='o')
axes[0].set_title('Train MSE'); axes[0].set_xlabel('Epoch')

axes[1].plot(history_frozen['epoch'], history_frozen['test_rmse'], marker='o', color='orange')
axes[1].set_title('Test RMSE'); axes[1].set_xlabel('Epoch')

axes[2].plot(history_frozen['epoch'], history_frozen['test_spearman'], marker='o', color='green', label='Spearman')
axes[2].plot(history_frozen['epoch'], history_frozen['test_pearson'],  marker='o', color='red',   label='Pearson')
axes[2].set_title('Correlation'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
preds_f, trues_f, rmse_f, rho_f, r_f = evaluate(model_frozen, test_loader)
print(f'Frozen — RMSE: {rmse_f:.2f} C,  Spearman: {rho_f:.3f},  Pearson: {r_f:.3f}')

## 6. Experiment 2 — Fine-tune Last 2 Transformer Layers

Now actual deep learning: gradients flow back into ESM-2's last 2 transformer layers. Use a *much smaller* learning rate for the ESM part so we don't destroy the pretrained weights.

In [ ]:
torch.manual_seed(42)
model_ft = ESM2Regressor(n_unfrozen_layers=2).to(device)

# Two parameter groups with different learning rates
esm_params = [p for n, p in model_ft.named_parameters() if 'esm' in n and p.requires_grad]
head_params = [p for n, p in model_ft.named_parameters() if 'head' in n]
opt = torch.optim.AdamW([
    {'params': esm_params,  'lr': 1e-5},   # tiny LR for pretrained layers
    {'params': head_params, 'lr': 1e-3},   # normal LR for new head
])

loss_fn = nn.MSELoss()
history_ft = []
n_epochs = 5
for epoch in range(n_epochs):
    model_ft.train(); t0 = time.time(); losses = []
    for enc, tms in train_loader:
        enc = {k: v.to(device) for k, v in enc.items()}
        tms = tms.to(device)
        opt.zero_grad()
        loss = loss_fn(model_ft(enc), tms)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    _, _, rmse, rho, r = evaluate(model_ft, test_loader)
    history_ft.append({'epoch': epoch, 'train_mse': np.mean(losses),
                       'test_rmse': rmse, 'test_spearman': rho, 'test_pearson': r})
    print(f'epoch {epoch}: train_mse={np.mean(losses):.2f}  test_rmse={rmse:.2f}  '
          f'spearman={rho:.3f}  ({time.time()-t0:.0f}s)')
history_ft = pd.DataFrame(history_ft)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].plot(history_ft['epoch'], history_ft['train_mse'], marker='o')
axes[0].set_title('Train MSE'); axes[0].set_xlabel('Epoch')

axes[1].plot(history_ft['epoch'], history_ft['test_rmse'], marker='o', color='orange')
axes[1].set_title('Test RMSE'); axes[1].set_xlabel('Epoch')

axes[2].plot(history_ft['epoch'], history_ft['test_spearman'], marker='o', color='green', label='Spearman')
axes[2].plot(history_ft['epoch'], history_ft['test_pearson'],  marker='o', color='red',   label='Pearson')
axes[2].set_title('Correlation'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
preds_ft, trues_ft, rmse_ft, rho_ft, r_ft = evaluate(model_ft, test_loader)
print(f'Fine-tuned — RMSE: {rmse_ft:.2f} C,  Spearman: {rho_ft:.3f},  Pearson: {r_ft:.3f}')

## 7. Compare the Two Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(history_frozen['epoch'], history_frozen['test_rmse'], 'o-', label='Frozen')
axes[0].plot(history_ft['epoch'],     history_ft['test_rmse'],     's-', label='Fine-tuned')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('Test RMSE (C)'); axes[0].legend()
axes[0].set_title('Test RMSE over training')

axes[1].plot(history_frozen['epoch'], history_frozen['test_spearman'], 'o-', label='Frozen')
axes[1].plot(history_ft['epoch'],     history_ft['test_spearman'],     's-', label='Fine-tuned')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('Test Spearman'); axes[1].legend()
axes[1].set_title('Test correlation over training')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, preds, trues, rmse, rho, name in [
    (axes[0], preds_f,  trues_f,  rmse_f,  rho_f,  'Frozen'),
    (axes[1], preds_ft, trues_ft, rmse_ft, rho_ft, 'Fine-tuned'),
]:
    ax.scatter(trues, preds, alpha=0.4, s=12)
    lim = [min(trues.min(), preds.min()), max(trues.max(), preds.max())]
    ax.plot(lim, lim, 'r--', lw=1)
    ax.set_xlabel('True Tm (C)'); ax.set_ylabel('Predicted Tm (C)')
    ax.set_title(f'{name}: RMSE={rmse:.2f}, Spearman={rho:.3f}')
plt.tight_layout(); plt.show()

## 8. Where Does Each Model Make Mistakes?

In [ ]:
errors = pd.DataFrame({
    'sequence': test_df['sequence'].values[:len(trues_ft)],
    'true_Tm':  trues_ft,
    'pred_frozen': preds_f,
    'pred_ft': preds_ft,
    'abs_err_frozen': np.abs(preds_f  - trues_f),
    'abs_err_ft':     np.abs(preds_ft - trues_ft),
})

print('Worst frozen-model predictions:')
print(errors.nlargest(5, 'abs_err_frozen')[['true_Tm', 'pred_frozen', 'abs_err_frozen']].round(2))
print('\nWorst fine-tuned predictions:')
print(errors.nlargest(5, 'abs_err_ft')[['true_Tm', 'pred_ft', 'abs_err_ft']].round(2))
print('\nLargest improvement from fine-tuning (by |error| reduction):')
errors['improvement'] = errors['abs_err_frozen'] - errors['abs_err_ft']
print(errors.nlargest(5, 'improvement')[['true_Tm', 'pred_frozen', 'pred_ft', 'improvement']].round(2))

## Reflection Questions

1. **Did fine-tuning help?** By how much? Is the gain worth the extra GPU time?
2. **Why two learning rates?** What would happen if we used `lr=1e-3` for the ESM-2 layers as well? (Hint: the pretrained weights would be destroyed.)
3. **Sequence length matters** — we truncated at 400 AA. Some real proteins are 1000+ residues. How might truncation bias which proteins the model gets right?
4. **Biology check** — pick a sequence the model predicted very poorly. Is it from a thermophilic organism? An unusually long/short protein? Does anything stand out?
5. **Scaling** — would unfreezing 4 layers help more than 2? Try if you have GPU credits.

**Phase 3 B2 milestone:** Spearman ≥ 0.5 on test set, with a clear comparison plot showing fine-tuning vs frozen, and an interpretation of when fine-tuning was worth it.

# Phase 2 Track B 進階：ESM2 Fine-tuning 熱穩定性預測

## 任務概述

- **資料集**：FLIP Meltome mixed split（27,951 條蛋白質，來自多物種）
- **目標**：從序列預測熔點 Tm（°C）— regression 任務
- **模型**：ESM2（預訓練蛋白質語言模型）+ regression head
- **核心概念**：Transfer learning、Freeze/Unfreeze、Fine-tuning

---

## 資料載入

FLIP repo 的資料以 zip 打包，需先解壓：

```python
FLIP_ZIP_URL = 'https://github.com/J-SNACKKB/FLIP/raw/main/splits/meltome/splits.zip'
# 解壓後讀取 splits/mixed_split.csv
```

欄位：`sequence`（胺基酸序列）、`target`（Tm °C）、`set`（train/test）

---

## 資料準備：Dataset → DataLoader → Tokenizer

### 為什麼需要這三層？

| 元件 | 功能 | 類比 |
|---|---|---|
| `Dataset` | 把 DataFrame 包裝成 PyTorch 格式 | 花名冊 |
| `DataLoader` | 自動分批、打亂順序 | 每次叫 8 個學生上來 |
| `Tokenizer` | 把胺基酸字母轉成數字 token | 翻譯官 |

### Collate 函數的作用

蛋白質序列長度不同，需要統一成矩陣：
- **Padding**：短序列補 `[PAD]` token
- **Truncation**：超過 `MAX_LEN`（400 AA）的序列截斷

```python
enc = tokenizer(seqs, padding=True, truncation=True, max_length=MAX_LEN)
```

---

## 模型架構：ESM2Regressor

```
輸入序列
    ↓ Tokenizer
ESM2 主體（特徵提取，frozen 或部分 unfrozen）
    ↓ Masked mean pooling（(B, L, D) → (B, D)）
Regression Head：Linear → ReLU → Dropout → Linear(1)
    ↓
預測 Tm 值
```

### Masked Mean Pooling

ESM2 對每個胺基酸位置都輸出一個向量。Pooling 將整條蛋白質壓成一個向量：

```python
pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
```

`attention_mask` 標記哪些位置是真實序列（1）、哪些是 padding（0），確保 padding 不影響平均。

---

## Transfer Learning：Freeze / Unfreeze

### 核心概念

ESM2 預訓練於數億條蛋白質序列，已編碼蛋白質通用知識。

| 策略 | 做法 | 適合情境 |
|---|---|---|
| 全凍結 | 只訓練 head | 資料少、快速驗證 |
| 部分解凍 | 解凍最後 N 層 + head | 有一定資料量、任務特定 |
| 全解凍 | 所有層都更新 | 資料量大（通常不建議） |

### 為什麼只解凍最後幾層？

```
前面幾層 → 通用特徵（胺基酸化學性質）← 不動
中間幾層 → 結構特徵（二級結構、motif）← 不動
最後幾層 → 任務相關特徵              ← 微調
```

### Catastrophic Forgetting

如果用太大的 learning rate 更新預訓練層，模型會「忘掉」預訓練知識：

```python
# 正確：兩組不同 lr
opt = AdamW([
    {'params': esm_params,  'lr': 1e-5},   # ESM 層：極小 lr
    {'params': head_params, 'lr': 1e-3},   # Head：正常 lr
])
```

---

## 訓練迴圈

```python
for enc, tms in train_loader:
    opt.zero_grad()              # 清空上一批梯度
    loss = loss_fn(model(enc), tms)   # MSE loss
    loss.backward()              # 反向傳播，計算梯度
    opt.step()                   # AdamW 更新參數
```

---

## 評估指標

| 指標 | 意思 | 重要性 |
|---|---|---|
| **RMSE** | 預測 Tm 差幾度 | 絕對準確度 |
| **Spearman ρ** | 排名順序對不對 | 蛋白質工程最重要 |
| **Pearson r** | 數值線性相關 | 輔助參考 |

Spearman 最重要，因為實際應用是「找出最穩定的突變體」，不需要精確 Tm 數值。

---

## 實驗結果（5 epochs）

| 模型 | Test RMSE | Spearman |
|---|---|---|
| Frozen（只訓練 head） | ~8.9°C | ~0.42 |
| Fine-tuned（解凍最後 2 層） | ~8.4°C | ~0.47 |

Fine-tuning 提升約 12%，且收斂更快（起點就更高）。

---

## 錯誤分析

**最難預測的蛋白質特徵：**
- Tm > 85°C 的極端耐熱蛋白質（嗜熱菌來源）
- 訓練資料中比例少
- 序列特徵與一般蛋白質差異大

**Regression to the mean：**  
Regression 模型傾向於往訓練集的平均值預測，對極端值（極高/極低 Tm）誤差特別大。

**截斷偏差：**  
MAX_LEN=400 AA，但許多耐熱蛋白質序列更長，C 端資訊被截斷可能影響預測。

---

## Reflection 重點

1. Fine-tuning 有效，但需要更多 epoch 才能看到完整差距
2. 兩組 learning rate 是防止 catastrophic forgetting 的標準做法
3. 截斷偏差對長蛋白質（尤其嗜熱菌）不利
4. 解凍更多層不一定更好，需用 validation set 選擇最佳 `n_unfrozen_layers`

---

## 與其他 Phase 2 Track 的連結

| Track | 特徵表示 | 模型 |
|---|---|---|
| A（DNA k-mer） | k-mer 頻率 | Random Forest |
| C（分子溶解度） | Morgan fingerprint | Random Forest |
| **B（蛋白質 Tm）** | **ESM2 embedding** | **Neural network** |
| D（scRNA-seq） | 基因表現矩陣 | Leiden clustering |

Track B 的核心進步：從「手工設計特徵」（k-mer、fingerprint）→「模型自動學特徵」（ESM2 embedding）。